# Notebook 05 — Customer Profiling & Business Insights
**Project:** Credit Card Customer Profiling & Segmentation  
**Goal:** Deeply analyse each cluster, assign business labels, and translate findings into actionable marketing strategies.

---
| Step | Action |
|------|--------|
| 1 | Cluster mean profiles |
| 2 | Feature heatmap |
| 3 | Key metric comparisons |
| 4 | Business labels & strategies |
| 5 | Final summary table |

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', font_scale=1.05)
plt.rcParams['figure.dpi'] = 120

from src.data_loader import load_raw_data
from src.preprocessing import preprocess
from src.config import RANDOM_STATE, N_CLUSTERS, REPORTS_DIR

palette = ['#2196F3','#F44336','#4CAF50','#FF9800']
segment_names = {
    0: 'Low Activity',
    1: 'Active Transactors',
    2: 'VIP / High Spenders',
    3: 'Cash Advance Heavy'
}

df_scaled, df_unscaled = preprocess(load_raw_data(), save=False)
km = KMeans(n_clusters=N_CLUSTERS, random_state=RANDOM_STATE, n_init=10)
df_unscaled['Cluster'] = km.fit_predict(df_scaled)
df_unscaled['Segment'] = df_unscaled['Cluster'].map(segment_names)
print('Cluster assignment complete.')
df_unscaled[['BALANCE','PURCHASES','CASH_ADVANCE','CREDIT_LIMIT','Cluster','Segment']].head(8)

## 1. Cluster Mean Profiles

In [ ]:
key_cols = ['BALANCE','PURCHASES','CASH_ADVANCE','CREDIT_LIMIT',
            'PAYMENTS','PRC_FULL_PAYMENT','PURCHASES_TO_LIMIT_RATIO',
            'CASH_ADVANCE_TO_CREDIT_RATIO','MONTHLY_AVG_PURCHASE','BALANCE_TO_CREDIT_RATIO']
key_cols = [c for c in key_cols if c in df_unscaled.columns]

profile = df_unscaled.groupby('Cluster')[key_cols].mean().round(2)
profile.index = [f"Cluster {k} — {segment_names[k]}" for k in profile.index]
profile

## 2. Feature Heatmap

In [ ]:
profile_norm = (profile - profile.min()) / (profile.max() - profile.min() + 1e-9)

fig, ax = plt.subplots(figsize=(14, 5))
sns.heatmap(profile_norm.T, annot=True, fmt='.2f', cmap='YlOrRd',
            linewidths=0.5, ax=ax, cbar_kws={'shrink': 0.7})
ax.set_title('Customer Segment Profiles (Normalised 0–1)', fontsize=13, fontweight='bold')
ax.set_xlabel('Segment')
ax.set_yticklabels(ax.get_yticklabels(), fontsize=8)
ax.set_xticklabels(ax.get_xticklabels(), fontsize=8, rotation=15, ha='right')
plt.tight_layout()
plt.show()

## 3. Key Metric Comparisons

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
metrics = [
    ('BALANCE', 'Average Balance ($)'),
    ('PURCHASES', 'Average Purchases ($)'),
    ('CASH_ADVANCE', 'Average Cash Advance ($)'),
    ('PRC_FULL_PAYMENT', 'Full Payment Rate'),
]

for ax, (col, title) in zip(axes.flatten(), metrics):
    vals = df_unscaled.groupby('Cluster')[col].mean()
    seg_lbls = [f'C{k}\n{segment_names[k][:12]}' for k in vals.index]
    bars = ax.bar(seg_lbls, vals.values, color=palette[:len(vals)], edgecolor='white', alpha=0.88)
    for bar, v in zip(bars, vals.values):
        label = f'${v:,.0f}' if col != 'PRC_FULL_PAYMENT' else f'{v:.2f}'
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.02,
                label, ha='center', fontsize=9, fontweight='bold')
    ax.set_title(title, fontweight='bold')
    ax.set_ylabel('Mean Value')

plt.suptitle('Key Metrics Comparison Across Segments', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Business Interpretation & Marketing Strategies

| Cluster | Segment | Size | Key Profile | Marketing Strategy |
|---|---|---|---|---|
| 0 | **Low Activity** | 3,917 (43.8%) | Low balance, low purchases, minimal engagement | Re-engagement offers, cashback on first purchases, spend incentives |
| 1 | **Active Transactors** | 3,475 (38.8%) | High purchases, moderate credit limit, regular payments | Premium rewards, travel miles, higher credit limit upgrades |
| 2 | **VIP / High Spenders** | 278 (3.1%) | Very high purchases ($9,389), high credit limit ($9,937), high payments | Concierge services, exclusive VIP cards, relationship manager |
| 3 | **Cash Advance Heavy** | 1,280 (14.3%) | Very high cash advance ($4,338), high balance, low full payment | Debt consolidation products, financial counselling, lower APR offers |

### Key Insights
1. **VIP High Spenders** (3%) generate disproportionate revenue — white-glove retention is critical
2. **Cash Advance Heavy** (14%) are at financial risk — proactive outreach can reduce default
3. **Low Activity** (44%) is the largest segment — even small activation improves revenue significantly
4. **Active Transactors** (39%) are ideal rewards card candidates — high spend, regular payments

In [ ]:
# Save final summary
summary = df_unscaled.groupby(['Cluster','Segment'])[key_cols].mean().round(2)
summary['Customer_Count'] = df_unscaled.groupby('Cluster').size()
summary.to_csv(REPORTS_DIR / 'cluster_summary.csv')
print('Saved: reports/cluster_summary.csv')
summary